Regularization LAB

Dataset: `california_housing.csv`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline


# Suppress warnings:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

np.set_printoptions(precision=3, suppress=True)

Pada sel berikut, kita akan memuat dataset dan mendefinisikan beberapa fungsi yang digunakan untuk membuat grafik.

In [ ]:
np.random.seed(72018)



def to_2d(array):
    return array.reshape(array.shape[0], -1)


    
def plot_exponential_data():
    data = np.exp(np.random.normal(size=1000))
    plt.hist(data)
    plt.show()
    return data
    
def plot_square_normal_data():
    data = np.square(np.random.normal(loc=5, size=1000))
    plt.hist(data)
    plt.show()
    return data

Load Dataset

In [1]:
import pandas as pd

housing_data = pd.read_csv("data/california_housing.csv")
print("Dataset shape:", housing_data.shape)
print("Columns:", housing_data.columns.tolist())

Dataset shape: (20640, 9)
Columns: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedHouseVal']


In [1]:
housing_data.head()

NameError: name 'housing_data' is not defined

Data Standardization

Standardization adalah proses mengubah skala setiap fitur agar memiliki mean 0 dan standard deviation 1.

Kita dapat menggunakan `StandardScaler` dari scikit-learn untuk melakukan proses ini.

In [ ]:
y_col = "MedHouseVal"

X = housing_data.drop(y_col, axis=1)
y = housing_data[y_col]

In [ ]:
from sklearn.preprocessing import StandardScaler

s = StandardScaler()
X_ss = s.fit_transform(X)

Exercise

Periksa hasil standardization. Nilai mean setiap fitur seharusnya mendekati 0 dan standard deviation mendekati 1.

In [ ]:
# Petunjuk:

a = np.array([[1, 2, 3], 
              [4, 5, 6]]) 
print(a) # 2 rows, 3 columns

In [ ]:
a.mean(axis=0) # mean along the *columns*

In [ ]:
a.mean(axis=1) # mean along the *rows*

In [ ]:
### BEGIN SOLUTION
X2 = np.array(X)
man_transform = (X2-X2.mean(axis=0))/X2.std(axis=0)
np.allclose(man_transform, X_ss)
### END SOLUTION

Coefficients with and without Standardization

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lr = LinearRegression()

y_col = "MedHouseVal"

X = housing_data.drop(y_col, axis=1)
y = housing_data[y_col]

In [ ]:
lr.fit(X, y)
print(lr.coef_) # nilai minimum pada dataset sebelumnya sekitar -18

#### Discussion

Koefisien model memiliki skala yang sangat berbeda. Apakah perbedaan skala ini selalu menjadi masalah? Jelaskan alasanmu.

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
s = StandardScaler()
X_ss = s.fit_transform(X)

In [ ]:
lr2 = LinearRegression()
lr2.fit(X_ss, y)
print(lr2.coef_) # koefisien sekarang berada pada skala yang sama

Exercise

Berdasarkan hasil tersebut, fitur mana yang tampak paling berpengaruh? Pertanyaan ini memang dapat ditafsirkan dengan lebih dari satu cara. Ke arah mana fitur tersebut memengaruhi nilai `y`?

Hint: Kita dapat memasangkan nama kolom DataFrame `df` dengan koefisien model `model` menggunakan:

```python
dict(zip(df.columns.values, model.coef_))
```

In [ ]:
### BEGIN SOLUTION
pd.DataFrame(zip(X.columns, lr2.coef_)).sort_values(by=1)
### END SOLUTION

Jika hanya melihat besar koefisien yang telah distandardisasi, fitur dengan nilai koefisien absolut terbesar dapat dianggap paling berpengaruh menurut ukuran tersebut. Namun, besar koefisien saja tidak cukup untuk menyimpulkan signifikansi statistik suatu fitur.

Lasso with and without Standardization

Kita telah membahas Lasso dalam perkuliahan. Mari kita tinjau kembali:

1. Apa perbedaan Lasso dengan Linear Regression biasa?
2. Apakah standardization lebih atau kurang penting untuk Lasso dibandingkan Linear Regression? Mengapa?

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.preprocessing import PolynomialFeatures

Creating Polynomial Features

Dokumentasi `PolynomialFeatures` tersedia di [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html).

Polynomial features menambahkan fitur berbentuk pangkat dan kombinasi antarfitur.

In [ ]:
pf = PolynomialFeatures(degree=2, include_bias=False,)
X_pf = pf.fit_transform(X)

Kita menggunakan `include_bias=False` karena `Lasso` sudah menyertakan intercept secara default.

In [ ]:
X_pf_ss = s.fit_transform(X_pf)

Lasso

Dokumentasi `Lasso` tersedia di [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html).

In [ ]:
las = Lasso()
las.fit(X_pf_ss, y)
las.coef_ 

Exercise

Bandingkan dua hal berikut:

- Sum of absolute coefficient values.
- Number of coefficients equal to zero.

Bandingkan model Lasso dengan `alpha=0.1` dan `alpha=1`.

Sebelum menjalankan latihan, jawab pertanyaan berikut masing-masing dalam satu kalimat:

- Model mana yang kamu perkirakan memiliki jumlah nilai absolut koefisien lebih besar?
- Model mana yang kamu perkirakan memiliki lebih banyak koefisien bernilai nol?

In [ ]:
### BEGIN SOLUTION
las01 = Lasso(alpha = 0.1)
las01.fit(X_pf_ss, y)
print('sum of absolute coefficient values:', abs(las01.coef_).sum() )
print('number of coefficients not equal to 0:', (las01.coef_!=0).sum())

In [ ]:
las1 = Lasso(alpha = 1)
las1.fit(X_pf_ss, y)
print('sum of absolute coefficient values:',abs(las1.coef_).sum() )
print('number of coefficients not equal to 0:',(las1.coef_!=0).sum())
### END SOLUTION

Dengan regularization yang lebih kuat (nilai `alpha` lebih tinggi), penalti untuk koefisien besar juga meningkat. Akibatnya, besar koefisien cenderung mengecil dan lebih banyak koefisien dapat menjadi nol.

Exercise: $R^2$

Hitung nilai $R^2$ setiap model pada dataset yang sama dengan dataset yang digunakan untuk melatih model, tanpa train-test split.

Kita dapat mengimpor `r2_score` seperti berikut:

```python
from sklearn.metrics import r2_score
```

In [ ]:
### BEGIN SOLUTION
from sklearn.metrics import r2_score
r2_score(y,las.predict(X_pf_ss))
### END SOLUTION

Discussion

Apakah regularization selalu meningkatkan performa model jika evaluasi dilakukan pada dataset yang sama dengan dataset pelatihan? Jelaskan.

Using a Train-Test Split

Discussion

Apakah ada masalah dengan pendekatan yang telah kita lakukan sejauh ini?

Hint: Perhatikan cara kita melakukan feature scaling. Untuk evaluasi yang benar, parameter scaler harus dipelajari dari data training saja, lalu digunakan untuk mentransformasi data testing.

Diskusikan bersama kelompok yang terdiri dari dua atau tiga orang.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_pf, y, test_size=0.3, 
                                                    random_state=72018)

In [ ]:
X_train_s = s.fit_transform(X_train)
las.fit(X_train_s, y_train)
X_test_s = s.transform(X_test)
y_pred = las.predict(X_test_s)
r2_score(y_test, y_pred)

In [ ]:
X_train_s = s.fit_transform(X_train)
las01.fit(X_train_s, y_train)
X_test_s = s.transform(X_test)
y_pred = las01.predict(X_test_s)
r2_score(y_test, y_pred)

### Exercise

#### Part 1

Lakukan prosedur yang sama menggunakan Lasso dengan:

- `alpha=0.001`
- `max_iter=100000` untuk membantu memastikan model mencapai konvergensi.

Hitung nilai $R^2$ model tersebut.

Kamu boleh menyalin kode dari sel sebelumnya, tetapi tuliskan komentar satu kalimat di atas setiap baris kode untuk menjelaskan alasan langkah tersebut dilakukan.

#### Part 2

Lakukan prosedur yang sama menggunakan Linear Regression, kemudian hitung nilai $R^2$ model tersebut.

#### Part 3

Bandingkan jumlah nilai absolut koefisien dan jumlah koefisien bernilai nol dari kedua model. Berdasarkan ukuran tersebut, model mana yang memberikan gambaran hubungan antara features dan target yang lebih sederhana?

In [ ]:
### BEGIN SOLUTION

# Part 1

# Decreasing regularization and ensuring convergence
las001 = Lasso(alpha = 0.001, max_iter=100000)

# Transforming training set to get standardized units
X_train_s = s.fit_transform(X_train)

# Fitting model to training set
las001.fit(X_train_s, y_train)

# Transforming test set using the parameters defined from training set
X_test_s = s.transform(X_test)

# Finding prediction on test set
y_pred = las001.predict(X_test_s)

# Calculating R² score
print("r2 score for alpha = 0.001:", r2_score(y_test, y_pred))


# Part 2

# Using vanilla Linear Regression
lr = LinearRegression()

# Fitting model to training set
lr.fit(X_train_s, y_train)

# predicting on test set
y_pred_lr = lr.predict(X_test_s)

# Calculating R² score
print("r2 score for Linear Regression:", r2_score(y_test,y_pred_lr))


# Part 3
print('Magnitude of Lasso coefficients:', abs(las001.coef_).sum())
print('Number of coeffients not equal to 0 for Lasso:', (las001.coef_!=0).sum())

print('Magnitude of Linear Regression coefficients:', abs(lr.coef_).sum())
print('Number of coeffients not equal to 0 for Linear Regression:', (lr.coef_!=0).sum())
### END SOLUTION

Sumber Dataset

Mendeley Data - California Housing Dataset

Simpan file sebagai `data/california_housing.csv` sebelum menjalankan notebook.